# Gemini PDF Citations & Grounding

This notebook demonstrates how to use Gemini's **File Search** tool to analyze PDF documents and receive grounded responses with specific citations (including page numbers).

In [14]:
# Load env variables and create client
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
import time

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-3-flash-preview" # Citations are best supported on 2.0+ models

## 1. Setup File Search Store

The **File Search** tool requires a store to hold and index your documents. This enables the model to perform efficient retrieval and provide accurate citations.

In [15]:
# 1. Create a File Search Store
store_display_name = "Earth Research Store"
store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(displayName=store_display_name)
)
print(f"Created store: {store.display_name} ({store.name})")

# 2. Upload and Index the PDF
pdf_path = project_root / "assets" / "pdfs" / "earth.pdf"
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

print(f"Uploading and indexing {pdf_path}...")

operation = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=store.name,
    file=pdf_path,
    config=types.UploadToFileSearchStoreConfig(displayName="Earth Data")
)

# 3. Wait for the indexing operation to complete
while not operation.done:
    print(".", end="", flush=True)
    time.sleep(2)
    operation = client.operations.get(operation)

if operation.error:
    raise RuntimeError(f"Indexing failed: {operation.error}")

print("\nIndexing complete!")

Created store: Earth Research Store (fileSearchStores/earth-research-store-0j8iw3p217la)
Uploading and indexing /Users/pyaephyowin/Documents/Anthropic Course/Claude/assets/pdfs/earth.pdf...
.
Indexing complete!


## 2. Generate Content with Citations

To get citations, we must enable the `file_search` tool in our request. The model will then use the indexed document to ground its response.

In [16]:
def ask_with_citations(prompt):
    # Configure the tool
    tool = types.Tool(
        file_search=types.FileSearch(
            file_search_store_names=[store.name]
        )
    )
    
    # Generate content with the tool enabled using the retry helper
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[tool]
        )
    )
    return response

prompt = "Tell me about the composition of Earth's atmosphere according to the document."
response = ask_with_citations(prompt)

print("\n--- Gemini Response ---")
print(response.text)


--- Gemini Response ---
According to the document, Earth's atmosphere is primarily composed of nitrogen and oxygen. The detailed composition by volume (for dry air) is as follows:

*   **Nitrogen:** 78.08%
*   **Oxygen:** 20.95%
*   **Argon:** 0.9340%
*   **Carbon dioxide:** 0.0415%
*   **Neon:** 0.00182%
*   **Helium:** 0.00052%
*   **Methane:** 0.00017%
*   **Krypton:** 0.00011%
*   **Hydrogen:** 0.00006%

Additionally, **water vapor** is present in variable amounts, typically less than or equal to 1%. It acts as a greenhouse gas and, along with carbon dioxide, helps maintain the surface conditions necessary for liquid water to exist on Earth.


## 3. Parsing Grounding Metadata

The response includes `grounding_metadata`, which links segments of the generated text to specific chunks of the source document, including page numbers.

In [18]:
metadata = response.candidates[0].grounding_metadata

if not metadata:
    print("No grounding metadata returned.")
else:
    if metadata.grounding_chunks:
        print("\n--- Sources used ---")
        for i, chunk in enumerate(metadata.grounding_chunks):
            if chunk.retrieved_context:
                ctx = chunk.retrieved_context
                # Note: page_number might not always be present depending on the document processing
                page = getattr(ctx, 'page_number', 'N/A')
                print(f"[{i}] Source: {ctx.title}, Page: {page}")

    if metadata.grounding_supports:
        print("\n--- Grounding Supports ---")
        for support in metadata.grounding_supports:
            print(f"\nSegment: '{support.segment.text.strip()}'")
            print(f"Supported by chunk indices: {support.grounding_chunk_indices}")


--- Sources used ---
[0] Source: Earth Data, Page: N/A
[1] Source: Earth Data, Page: N/A
[2] Source: Earth Data, Page: N/A
[3] Source: Earth Data, Page: N/A
[4] Source: Earth Data, Page: N/A

--- Grounding Supports ---

Segment: 'According to the document, Earth's atmosphere is primarily composed of nitrogen and oxygen.'
Supported by chunk indices: [0, 1]

Segment: '*   **Nitrogen:** 78.08%'
Supported by chunk indices: [1]

Segment: '*   **Oxygen:** 20.95%'
Supported by chunk indices: [1]

Segment: '*   **Argon:** 0.9340%'
Supported by chunk indices: [1]

Segment: '*   **Carbon dioxide:** 0.0415%'
Supported by chunk indices: [1]

Segment: '*   **Neon:** 0.00182%'
Supported by chunk indices: [1]

Segment: '*   **Helium:** 0.00052%'
Supported by chunk indices: [1]

Segment: '*   **Methane:** 0.00017%'
Supported by chunk indices: [1]

Segment: '*   **Krypton:** 0.00011%'
Supported by chunk indices: [1]

Segment: '*   **Hydrogen:** 0.00006%'
Supported by chunk indices: [1]

Segment: 'Addi

## 4. Cleanup (Optional)

It's good practice to delete stores that are no longer needed to manage your resources effectively.

In [20]:
# Delete the store and its indexed documents to clean up
client.file_search_stores.delete(
    name=store.name,
    config=types.DeleteFileSearchStoreConfig(force=True),
)
print(f"Deleted store: {store.name}")

Deleted store: fileSearchStores/earth-research-store-0j8iw3p217la
